# Unsupervised Deep ContourFlow (DCF)

Evolve an initial contour on a single image using VGG16 feature contrast (inside vs outside the contour), with no labels or training required.

**Steps:** load image → initialize contour → predict → visualize evolution

In [ ]:
import sys
import os
from pathlib import Path
path_dcf = str(Path(os.path.abspath('.')).resolve().parent)
sys.path.append(path_dcf)
from deep_contourflow import UnsupervisedDCF as DCF
from torch_contour import CleanContours
import numpy as np
import matplotlib.pyplot as plt
import cv2
from deep_contourflow.features import define_contour_init
import torch
from deep_contourflow.visualization import plot_contour_evolution

viz_dir = os.path.join(path_dcf, "visualisations")
os.makedirs(viz_dir, exist_ok=True)

### Load image

In [ ]:
filename = "pineapple.jpg"
height = 512
img = plt.imread(os.path.join(path_dcf, "data", filename))
img = cv2.resize(img, (height, height), interpolation=cv2.INTER_AREA).astype(np.uint8)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
tensor = (torch.tensor(np.moveaxis(img, -1, 0)[None]) / 255).to(device)

fig, ax = plt.subplots(figsize=(5, 5))
fig.patch.set_facecolor("#FAF9F6")
ax.imshow(img)
ax.axis("off")
ax.set_title(filename, fontsize=11, fontweight=500, color="#3A3A3A", pad=10, loc="left")
plt.tight_layout(pad=0.4)
fig.savefig(os.path.join(viz_dir, f"input_{filename}.png"), dpi=180, bbox_inches="tight", facecolor="#FAF9F6")
plt.show()

### Instantiate the unsupervised DCF model

In [ ]:
dcf = DCF(
    model="vgg16",
    n_epochs=100,                    # max gradient steps
    learning_rate=1e-2,              # gradient step size
    area_force=1e-3,                 # weight of the area regularization term
    sigma=5e-1,                       # Gaussian smoothing strength on the gradient
    clip=1e-1,                       # max contour displacement per step (normalized)
    early_stopping_patience=100,
    early_stopping_threshold=1e-6,
    use_mixed_precision=True,        # faster on GPU
    do_apply_grabcut=False,
)

### Initialize contour

In [ ]:
nb_nodes = 200
contour_init, mask = define_contour_init(n=height, shape="circle", size=0.5)

c = CleanContours()
contour_init = c.interpolate(contour_init, nb_nodes).clip(0, 1)
contour_init = torch.tensor(contour_init)[None, None].to(torch.float32).to(device)

### Predict and visualize contour evolution

In [ ]:
contours, loss_history, final_contours = dcf.predict(tensor, contour_init)

fig = plot_contour_evolution(
    img, contours, loss_history,
    save_path=os.path.join(viz_dir, f"contour_evolution_{filename}.png"),
)
plt.show()